In [8]:
import requests
import pandas as pd
import uuid

# Basis configuratie gebaseerd op de technische documentatie
BASE_URL = "https://api.ah.nl"
STORE_ID = "1558"  # Dit is het unieke ID voor AH Woenselse Markt Eindhoven

# Verplichte headers om de AH app na te bootsen
headers = {
    "User-Agent": "Appie/9.28 (iPhone17,3; iPhone; CPU OS 26_1 like Mac OS X)",
    "x-application": "AHWEBSHOP",
    "x-clientname": "appie-ios",
    "x-": "9.28",
    "x-fraud-detection-installation-id": str(uuid.uuid4()), # Een unieke ID per sessie
    "Content-Type": "application/json",
    "Accept": "application/json"
}


In [9]:
def get_anonymous_token():
    auth_url = f"{BASE_URL}/mobile-auth/v1/auth/token/anonymous"
    payload = {"clientId": "appie-ios"}
    
    response = requests.post(auth_url, json=payload, headers=headers)
    response.raise_for_status() # Geeft een foutmelding als het misgaat
    
    token_data = response.json()
    return token_data['access_token']

# Activeer de sleutel voor alle volgende verzoeken
access_token = get_anonymous_token()
headers["Authorization"] = f"Bearer {access_token}"
print("Handshake succesvol: Token opgehaald.")

Handshake succesvol: Token opgehaald.


In [16]:
bargain_query = """
query GetBargains($storeId: String!) {
  bargainItems(storeId: $storeId) {
    categoryTitle  # <--- Deze voegt de categorie (zoals Vlees) toe
    product {
      title
      brand
      salesUnitSize
    }
    bargainPrice {
      priceWas
      priceNow
    }
    markdown {
      markdownPercentage
      markdownExpirationDate
    }
    stock
  }
}
"""

def fetch_laatste_kans(store_id):
    url = f"{BASE_URL}/graphql"
    
    graphql_headers = headers.copy()
    graphql_headers.update({
        "x-apollo-operation-name": "GetBargains",
        "x-apollo-operation-type": "query",
        "apollographql-client-name": "nl.ah.Appie-apollo-ios",
        "apollographql-client-version": "9.28-260102201630"
    })
    
    payload = {
        'query': bargain_query, 
        'variables': {'storeId': store_id},
        'operationName': 'GetBargains'
    }
    
    response = requests.post(url, json=payload, headers=graphql_headers)
    data = response.json()
    
    if 'errors' in data:
        print("❌ GraphQL Foutmelding gevonden:")
        for error in data['errors']:
            print(f" - {error.get('message')}")
        return None
        
    return data.get('data', {}).get('bargainItems')

# Haal de ruwe data opnieuw op
raw_items = fetch_laatste_kans(STORE_ID)

if raw_items:
    print(f"✅ Succes! {len(raw_items)} producten gevonden op de Woenselse Markt.")
else:
    print("❌ Geen data ontvangen. Controleer de output hierboven.")

✅ Succes! 143 producten gevonden op de Woenselse Markt.


In [18]:
# Gebruik json_normalize om geneste velden (zoals product.title) plat te slaan
df = pd.json_normalize(raw_items)

# Optioneel: Kolomnamen opschonen voor gemak
df.columns = [c.replace('product.', '').replace('bargainPrice.', '').replace('markdown.', '') for c in df.columns]

# Sorteer op de hoogste korting
df = df.sort_values(by='markdownPercentage', ascending=False)

# Toon de live status
display(df.head(10))

,categoryTitle,stock,priceWas,priceNow,markdownPercentage,markdownExpirationDate,title,brand,salesUnitSize
92,Bakkerij,1,3.29,0.99,70,2026-01-28,Dr. Oetker Kwarktaart eigen fruit bakmix,Dr. Oetker,210 g
83,"Zuivel, eieren",3,2.49,0.75,70,2026-01-26,Optimel Magere vla vanille,Optimel,1 l
73,Vleeswaren,3,1.99,0.60,70,2026-01-29,AH Duitse theeworst,AH,125 g
127,"Koek, snoep, chocolade",11,2.29,1.15,50,2026-02-01,Galler Wit kokosnoot,Galler,70 g
113,"Koek, snoep, chocolade",4,2.79,1.40,50,2026-01-25,Mentos Gum Sour strawberry,Mentos Gum,56 g
116,"Koek, snoep, chocolade",1,1.99,1.00,50,2026-01-25,Food2Smile Very berry,Food2Smile,90 g
115,"Koek, snoep, chocolade",1,3.99,2.00,50,2026-01-25,Stimorol Spearmint,Stimorol,101.5 g
114,"Koek, snoep, chocolade",2,2.99,1.50,50,2026-01-25,Fruittella Berries & cherry,Fruittella,200 g
126,"Koek, snoep, chocolade",14,4.19,2.10,50,2026-02-01,Lotus Biscoff Speculoos witte chocolade stukjes,Lotus Biscoff,180 g
110,"Koek, snoep, chocolade",10,2.79,1.40,50,2026-01-25,Mentos Gum Pure fresh strong euca menthol,Mentos Gum,56 g
